# Full Synthetic Dataset Generation

This notebook generates a complete synthetic dataset:
1. Load all components and configs
2. Generate N synthetic respondents
3. Run complete survey for all concepts
4. Export to Excel format
5. Save for validation

In [1]:
import sys
from pathlib import Path
import os
import json
from dotenv import load_dotenv
from datetime import datetime

sys.path.append(str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / '.env')

from src.persona.generator import PersonaGenerator
from src.llm.client import LLMClient
from src.ssr.embeddings import EmbeddingService
from src.ssr.rating_engine import RatingEngine
from src.scales.registry import create_default_scales
from src.survey.question_handler import QuestionHandler
from src.survey.survey_engine import SurveyEngine
from src.output.excel_formatter import ExcelFormatter
from src.logic.conditional_logic import create_standard_survey_logic

import warnings
warnings.filterwarnings('ignore')

## 1. Configuration

In [2]:
# ===== CONFIGURATION =====

# Number of synthetic respondents to generate
N_RESPONDENTS = 50  # Start small for testing, increase to 200 for full study

# Questions to ask (None = all from survey logic)
QUESTIONS_TO_ASK = ['B2', 'B3', 'B4', 'B6', 'B11', 'B12', 'B13', 'B14', 'B15', 'B16']
# For full survey, use: None

# Random seed for reproducibility
RANDOM_SEED = 42

# Output settings
OUTPUT_DIR = Path.cwd().parent / 'data/synthetic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = OUTPUT_DIR / f'synthetic_data_{N_RESPONDENTS}resp_{timestamp}.xlsx'

print("Configuration:")
print(f"  N Respondents: {N_RESPONDENTS}")
print(f"  Questions: {len(QUESTIONS_TO_ASK) if QUESTIONS_TO_ASK else 'All'}")
print(f"  Output: {OUTPUT_FILE}")

Configuration:
  N Respondents: 50
  Questions: 10
  Output: /Users/mark.stent/Projects/python/kantar-test/data/synthetic/synthetic_data_50resp_20251210_152713.xlsx


## 2. Load Concepts

In [3]:
# Load parsed concepts
concepts_file = Path.cwd().parent / 'data/parsed/concepts.json'

if concepts_file.exists():
    with open(concepts_file, 'r') as f:
        concepts = json.load(f)
    print(f"✓ Loaded {len(concepts)} concepts from parsed data")
else:
    # Create sample concepts if not parsed yet
    print("⚠️  Concepts not parsed yet. Using sample concepts.")
    print("   Run notebook 01 first to parse actual concepts.")
    
    concepts = [
        {
            'id': f'Concept {i}',
            'name': f'Test Concept {i}',
            'description': f'Sample concept {i} for testing',
            'price': '250 CZK'
        }
        for i in range(1, 9)
    ]
    print(f"✓ Created {len(concepts)} sample concepts")

for i, c in enumerate(concepts[:3], 1):
    print(f"  {i}. {c['name']}")

✓ Loaded 8 concepts from parsed data
  1. Concept 1: Christmas AR Message
  2. Concept 2: Elf Yourself
  3. Concept 3: AR Christmas Mini Game


## 3. Initialize All Components

In [4]:
print("Initializing components...")

# Core components
persona_generator = PersonaGenerator(random_seed=RANDOM_SEED)
print("  ✓ Persona generator")

llm_client = LLMClient(model="gpt-4o", temperature=0.5)
print("  ✓ LLM client (GPT-4o)")

embedding_service = EmbeddingService(model_id="text-embedding-3-small")
print("  ✓ Embedding service")

scale_registry = create_default_scales()
print(f"  ✓ Scale registry ({len(scale_registry.scales)} scales)")

rating_engine = RatingEngine(scale_registry, embedding_service)
print("  ✓ Rating engine")

# Preload anchor embeddings to reduce API calls
print("\nPreloading anchor embeddings...")
rating_engine.preload_scales()
print("  ✓ Anchors preloaded")

# Question handler
question_handler = QuestionHandler(llm_client, rating_engine)
print("  ✓ Question handler")

# Survey logic
survey_logic = create_standard_survey_logic()
print("  ✓ Survey logic")

# Survey engine
survey_engine = SurveyEngine(
    question_handler=question_handler,
    survey_logic=survey_logic,
    concepts=concepts
)
print("  ✓ Survey engine")

print("\n✅ All components initialized!")

Initializing components...
  ✓ Persona generator
  ✓ LLM client (GPT-4o)
  ✓ Embedding service
  ✓ Scale registry (13 scales)
  ✓ Rating engine

Preloading anchor embeddings...
  ✓ Anchors preloaded
  ✓ Question handler
  ✓ Survey logic
  ✓ Survey engine

✅ All components initialized!


## 4. Generate Synthetic Dataset

In [5]:
# Run the study
print(f"\nGenerating synthetic data for {N_RESPONDENTS} respondents...")
print(f"Testing {len(concepts)} concepts")
print(f"Asking {len(QUESTIONS_TO_ASK) if QUESTIONS_TO_ASK else 'all'} questions per concept")
print("\nThis may take several minutes depending on N and API speed...\n")

respondent_data_list = survey_engine.run_full_study(
    n_respondents=N_RESPONDENTS,
    persona_generator=persona_generator,
    concepts=concepts,
    questions_to_ask=QUESTIONS_TO_ASK,
    randomize_concepts=True,
    show_progress=True
)

print(f"\n✓ Generated data for {len(respondent_data_list)} respondents")

# Check for screened respondents
screened = [r for r in respondent_data_list if r.metadata.get('screened_out')]
valid = [r for r in respondent_data_list if not r.metadata.get('screened_out')]

print(f"  Valid: {len(valid)}")
print(f"  Screened out: {len(screened)}")


Generating synthetic data for 50 respondents...
Testing 8 concepts
Asking 10 questions per concept

This may take several minutes depending on N and API speed...

Generating 50 personas...


Running survey:   8%|▊         | 4/50 [06:29<1:14:33, 97.25s/it]


KeyboardInterrupt: 

## 5. Preview Sample Responses

In [ ]:
# Show sample responses
if valid:
    print("Sample Respondent Data:")
    print("="*80)
    
    sample = valid[0]
    print(f"\nRespondent: {sample.respondent_id}")
    print(f"Persona: {sample.persona.to_description()}")
    print(f"\nConcept Responses: {len(sample.concept_responses)}")
    
    if sample.concept_responses:
        first_concept = sample.concept_responses[0]
        print(f"\nFirst Concept: {first_concept.concept_name}")
        print(f"Position: {first_concept.position}")
        print(f"Responses collected: {len(first_concept.responses)}")
        
        # Show first few responses
        for q_id, response in list(first_concept.responses.items())[:3]:
            print(f"\n  {q_id}: {response.formatted_response}")
            if response.metadata.get('confidence'):
                print(f"       (confidence: {response.metadata['confidence']:.3f})")

## 6. Export to Excel

In [ ]:
# Format and save
print("\nFormatting and exporting to Excel...")

formatter = ExcelFormatter()
df = formatter.format_and_save(
    respondent_data_list=respondent_data_list,
    output_path=str(OUTPUT_FILE)
)

print(f"\n✓ Saved synthetic data: {OUTPUT_FILE}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

## 7. Preview Excel Output

In [ ]:
# Show sample of DataFrame
print("\nDataFrame Preview:")
print("="*80)

# Show demographic columns
demo_cols = ['SERIAL', 'Gender', 'AGE', '(AGEQUOTA) AGEBANDS']
available_demo = [c for c in demo_cols if c in df.columns]
print("\nDemographics:")
print(df[available_demo].head())

# Show sample question columns
question_cols = [c for c in df.columns if 'PRPURINT' in c or 'UNIQUENESS' in c]
if question_cols:
    print("\nSample Question Responses:")
    print(df[question_cols[:2]].head())

## 8. Basic Quality Checks

In [ ]:
import pandas as pd

print("Quality Checks:")
print("="*80)

# Check demographics
print("\n1. Age Distribution:")
print(f"   Min: {df['AGE'].min()}")
print(f"   Max: {df['AGE'].max()}")
print(f"   Mean: {df['AGE'].mean():.1f}")

print("\n2. Gender Distribution:")
print(df['Gender'].value_counts())

print("\n3. Age Band Distribution:")
if '(AGEQUOTA) AGEBANDS' in df.columns:
    print(df['(AGEQUOTA) AGEBANDS'].value_counts())

# Check question responses
pi_cols = [c for c in df.columns if 'PRPURINT' in c]
if pi_cols:
    print(f"\n4. Purchase Intent Response Rate:")
    for col in pi_cols[:2]:
        response_rate = (df[col].notna().sum() / len(df)) * 100
        print(f"   {col}: {response_rate:.1f}%")

## 9. Save Metadata

In [ ]:
# Save generation metadata
metadata = {
    'generation_timestamp': timestamp,
    'n_respondents': N_RESPONDENTS,
    'n_concepts': len(concepts),
    'questions_asked': QUESTIONS_TO_ASK or 'all',
    'random_seed': RANDOM_SEED,
    'valid_respondents': len(valid),
    'screened_respondents': len(screened),
    'output_file': str(OUTPUT_FILE),
    'models': {
        'llm': 'gpt-4o',
        'embedding': 'text-embedding-3-small',
        'temperature': 0.5
    }
}

metadata_file = OUTPUT_DIR / f'metadata_{timestamp}.json'
with open(metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Saved metadata: {metadata_file}")

## Summary

Synthetic dataset generation complete!

✅ **Generated** N synthetic respondents with realistic personas

✅ **Tested** all concepts with conditional logic

✅ **Exported** to Excel format matching ground truth structure

✅ **Quality Checked** - Demographics and responses look valid

**Output Files:**
- Synthetic data: `{OUTPUT_FILE}`
- Metadata: `{metadata_file}`

**Next Steps:**
- Run notebook 07 for validation
- Compare distributions to ground truth
- Calculate KL divergence, correlation, etc.
- Generate visualizations

**To Generate Full Dataset:**
1. Set `N_RESPONDENTS = 200` (or match ground truth)
2. Set `QUESTIONS_TO_ASK = None` (to ask all questions)
3. Run this notebook again
4. Note: Full run may take 30-60 minutes depending on API speed